<a href="https://colab.research.google.com/github/AfzalHossan-2005021/Advance-Data-Structures-and-Algorithms/blob/main/CODA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required packages
!apt-get install -y openslide-tools
!pip install openslide-python cellpose scikit-image matplotlib pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libopenslide0
Suggested packages:
  libtiff-tools
The following NEW packages will be installed:
  libopenslide0 openslide-tools
0 upgraded, 2 newly installed, 0 to remove and 35 not upgraded.
Need to get 104 kB of archives.
After this operation, 297 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenslide0 amd64 3.4.1+dfsg-5build1 [89.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openslide-tools amd64 3.4.1+dfsg-5build1 [13.8 kB]
Fetched 104 kB in 1s (116 kB/s)
Selecting previously unselected package libopenslide0.
(Reading database ... 126284 files and directories currently installed.)
Preparing to unpack .../libopenslide0_3.4.1+dfsg-5build1_amd64.deb ...
Unpacking libopenslide0 (3.4.1+dfsg-5build1) ...
Selecting previously unselected package openslide-tools.

In [2]:
!pip install -U gdown
import gdown

file_id = '1P0dRI8CRMu_kCaZ3s_tMnxCtt4RdpB5g'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'your_file.ndpi', quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1P0dRI8CRMu_kCaZ3s_tMnxCtt4RdpB5g
To: /content/your_file.ndpi
100%|██████████| 51.3M/51.3M [00:00<00:00, 73.2MB/s]


'your_file.ndpi'

In [ ]:
import openslide
import os
import numpy as np
from PIL import Image
from skimage.measure import regionprops
from cellpose import models, core
import csv

# Define paths and parameters
slide_path = "/content/your_file.ndpi"
tile_size = 1024
overlap = 128
stride = tile_size - overlap
output_csv = "centroids.csv"

# Open slide
slide = openslide.OpenSlide(slide_path)
width, height = slide.level_dimensions[0]

# Detect if GPU is available
use_gpu = core.use_gpu()

# Load Cellpose model
model = models.CellposeModel(gpu=use_gpu, model_type='nuclei')

# Create CSV output
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["tile", "x_global", "y_global"])

    # Tiling and nucleus detection
    for y in range(0, height, stride):
        for x in range(0, width, stride):
            region = slide.read_region((x, y), 0, (tile_size, tile_size)).convert("RGB")
            img_np = np.array(region)

            masks, _, _ = model.eval(img_np, diameter=None, channels=[0, 0])
            for prop in regionprops(masks):
                cy, cx = prop.centroid
                writer.writerow([f"tile_{x}_{y}.png", x + cx, y + cy])




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	linux 
python version: 	3.11.13 
torch version:  	2.6.0+cu124! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




100%|██████████| 1.15G/1.15G [00:08<00:00, 146MB/s]


In [ ]:
from google.colab import files
files.download("centroids.csv")

In [ ]:
import matplotlib.pyplot as plt

region = slide.read_region((0, 0), 0, (tile_size, tile_size)).convert("RGB")
img_np = np.array(region)
masks, _, _, _ = model.eval(img_np, diameter=None, channels=[0, 0])
centroids = [prop.centroid for prop in regionprops(masks)]

plt.imshow(img_np)
for y, x in centroids:
    plt.plot(x, y, 'r.', markersize=3)
plt.title(f"{len(centroids)} nuclei in tile (0,0)")
plt.axis("off")
plt.show()